# Linear Regression (OLS)

Plain OLS. Features are standardized inside a pipeline (scaler fit on training data only, then applied to test — no leakage). Standardizing doesn't change OLS predictions, but it makes the fitted coefficients directly comparable across features.

Same rolling-window pipeline as the other model notebooks — one `.parquet` read once and sliced by year, the trailing/cumulative window toggle, and a 4-column output (`permno, eom, target_w, prediction`) — with OLS as the estimator.

**Validation handling:** OLS has nothing to tune, so it trains on the combined train+valid years and predicts the test year. No leakage — the test year is always strictly after everything used to fit.

**Coefficient statistics are saved.** For every fold, the classical coefficient table (`coef, SE, t_stat, p_value`, 95% CI) is computed on the training design the model was fit on and written to `OLS_coef_stats.parquet`. Nothing is refit to produce it — it reuses the fitted pipeline's own scaler and coefficients — so downstream inference never has to re-run these models or re-read the feature panel.

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from numpy.linalg import pinv
from scipy.stats import t as t_dist

In [ ]:
# ---- configuration ---------------------------------------------------------
PARQUET_PATH = "US_GFD_FEATURES.parquet"
OUTPUT_PATH  = "OLS_predictions.parquet"
MODEL_DIR    = "OLS_models"   # fitted models saved here for coefficient analysis
COEF_STATS_PATH = "OLS_coef_stats.parquet"   # per-model coef / SE / t / p / CI table
CONF            = 0.95                        # confidence level for the stored CIs

FIRST_YEAR = 1984
LAST_YEAR  = 2025
N_TRAIN    = 5
N_VALID    = 5
N_TEST     = 1

# "trailing" -> fixed N_TRAIN+N_VALID-year train window sliding forward
# "cumulative" -> expanding train from FIRST_YEAR up to the test year
SCHEME = "trailing"    # or "cumulative"

ID_COLS = ["permno","eom","gvkey","iid","cusip","tic","tpci","exchg",
           "shrcd","exchcd","sic","naics","trade_eom","acc_eom",
           "acc_datadate","rdq","date_buy_t1","date_sell_t1","ret_1m",
           "target","prc","prc_buy_t1","prc_sell_t1","n_days",
           "acc_age_m","target_w"]
TARGET = "target_w"

In [ ]:
# ---- resolve feature list from schema (no data read) -----------------------
schema = pq.read_schema(PARQUET_PATH)
FEATURES = [c for c in schema.names if c not in ID_COLS]
READ_COLS = ["permno", "gvkey", "eom", TARGET] + FEATURES
print(f"{len(FEATURES)} feature columns")

In [ ]:
# ---- read once, slice by year ---------------------------------------------
import time
_t = time.time()
FULL = pq.read_table(PARQUET_PATH, columns=READ_COLS).to_pandas()
FULL["eom"] = pd.to_datetime(FULL["eom"])
FULL = FULL.astype({**{c: "float32" for c in FEATURES}, TARGET: "float32"})
FULL["_year"] = FULL["eom"].dt.year
print(f"read {len(FULL):,} rows x {len(FULL.columns)} cols in {time.time()-_t:.1f}s, "
      f"{FULL.memory_usage(deep=True).sum()/1e9:.2f} GB")

_na = FULL[FEATURES].isna().to_numpy().sum()
_inf = np.isinf(FULL[FEATURES].to_numpy()).sum()
print(f"feature NaNs: {_na:,} | Infs: {_inf:,} | target NaNs: {FULL[TARGET].isna().sum():,}")
if _na or _inf:
    FULL[FEATURES] = FULL[FEATURES].replace([np.inf, -np.inf], np.nan)
    FULL[FEATURES] = FULL[FEATURES].fillna(0.5).astype("float32")

def slice_years(y0, y1):
    sub = FULL[FULL["_year"].between(y0, y1)]
    return sub[sub[TARGET].notna()]

In [ ]:
# ---- window schedule (trailing vs cumulative) ------------------------------
def make_windows(first_year, last_year, n_tr, n_va, n_te, scheme=SCHEME):
    wins = []
    test_start = first_year + n_tr + n_va
    while test_start <= last_year:
        va = (test_start - n_va, test_start - 1)
        te = (test_start, min(test_start + n_te - 1, last_year))
        if scheme == "trailing":
            tr = (test_start - n_tr - n_va, test_start - n_va - 1)
        elif scheme == "cumulative":
            tr = (first_year, test_start - n_va - 1)
        else:
            raise ValueError(f"unknown SCHEME {scheme!r}")
        wins.append((tr, va, te))
        test_start += n_te
    return wins

WINDOWS = make_windows(FIRST_YEAR, LAST_YEAR, N_TRAIN, N_VALID, N_TEST, SCHEME)
print(f"scheme={SCHEME!r}: {len(WINDOWS)} windows | first {WINDOWS[0]} | last {WINDOWS[-1]}")

In [ ]:
# ---- fit OLS (standardize -> linear) on the full train+valid block ---------
def ols_coef_stats(pipe, X, y, feature_names, test_year, conf=CONF):
    """Classical coefficient table for the fitted model, computed on the TRAINING
    design it was fit on. There, beta IS the exact least-squares solution, so
    sigma^2 (X'X)^-1 is the exact textbook covariance. Nothing is refit: this
    reuses the pipeline's own scaler and fitted coefficients. That is why SE / t /
    p can be stored now and never need recomputing (or a model rerun) downstream."""
    scaler = pipe.named_steps["standardscaler"]
    lin    = pipe.named_steps["linearregression"]
    Xs   = np.asarray(scaler.transform(X), dtype=np.float64)     # same scaler the model uses
    yv   = np.asarray(y, dtype=np.float64)
    e    = yv - pipe.predict(X)                                  # residuals on the fit data
    n    = yv.size
    beta = np.concatenate([[float(lin.intercept_)], np.ravel(lin.coef_).astype(np.float64)])
    p    = beta.size                                             # intercept + features
    dof  = n - p
    Xd   = np.column_stack([np.ones(n), Xs])
    XtXi = pinv(Xd.T @ Xd)
    sig2 = float(e @ e) / dof
    se   = np.sqrt(np.clip(sig2 * np.diag(XtXi), 0.0, None))
    tval = np.divide(beta, se, out=np.full_like(beta, np.nan), where=se > 0)
    pval = 2.0 * t_dist.sf(np.abs(tval), dof)
    tcr  = t_dist.ppf(1 - (1 - conf) / 2, dof)
    return pd.DataFrame({
        "model_year": np.int32(test_year),
        "feature":    ["intercept"] + list(feature_names),
        "coef": beta, "abs_coef": np.abs(beta), "SE": se,
        "t_stat": tval, "p_value": pval,
        "ci_low": beta - tcr * se, "ci_high": beta + tcr * se,
        "n_obs": np.int64(n), "dof": np.int64(dof),
    })


def fit_predict_window(Xtr, ytr, Xte, Xtr_only, ytr_only, Xva, yva,
                       test_year=None, model_dir=None, feature_names=None):
    # standardize features (fit on train only), then plain OLS.
    import joblib
    m = make_pipeline(StandardScaler(), LinearRegression())
    m.fit(Xtr, ytr)
    if model_dir is not None:
        # the pipeline holds the scaler + fitted coefficients; the OLS step's
        # .coef_ is the (standardized) coefficient for each feature.
        joblib.dump(m, f"{model_dir}/ols_test{test_year}.joblib")
    # coefficient stats on the training design -- cheap here because Xtr is
    # already in memory, and stored so no rerun is ever needed.
    stats = ols_coef_stats(m, Xtr, ytr, feature_names, test_year)
    return m.predict(Xte), stats


In [ ]:
# ---- roll through windows, write predictions + coefficient stats -----------
import json
from pathlib import Path
Path(MODEL_DIR).mkdir(exist_ok=True)
json.dump(FEATURES, open(f"{MODEL_DIR}/feature_names.json", "w"))
writer = None
coef_stats = []
for (tr, va, te) in WINDOWS:
    train_df = slice_years(tr[0], va[1])     # train+valid block = training data
    test_df  = slice_years(te[0], te[1])     # the held-out test year

    Xtr = train_df[FEATURES].to_numpy(np.float32)
    ytr = train_df[TARGET].to_numpy(np.float32)
    Xte = test_df[FEATURES].to_numpy(np.float32)

    # for models that use early stopping (xgboost), split off the valid years;
    # others ignore va and just use the whole train block.
    va_mask = train_df["eom"].dt.year.between(*va)
    tr_mask = train_df["eom"].dt.year.between(*tr)
    Xtr_only = train_df.loc[tr_mask, FEATURES].to_numpy(np.float32)
    ytr_only = train_df.loc[tr_mask, TARGET].to_numpy(np.float32)
    Xva = train_df.loc[va_mask, FEATURES].to_numpy(np.float32)
    yva = train_df.loc[va_mask, TARGET].to_numpy(np.float32)

    yhat, stats = fit_predict_window(Xtr, ytr, Xte, Xtr_only, ytr_only, Xva, yva,
                                     te[0], MODEL_DIR, FEATURES)
    coef_stats.append(stats)

    out = test_df[["permno", "gvkey", "eom", TARGET]].copy()
    out["prediction"] = yhat.astype(np.float32)
    tbl = pa.Table.from_pandas(out, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PATH, tbl.schema)
    writer.write_table(tbl)
    print(f"test {te[0]}: train {tr}, valid {va} | "
          f"n_train={len(Xtr):,} n_test={len(Xte):,} | pred mean={yhat.mean():.5f}")

if writer is not None:
    writer.close()

# one combined coefficient-statistics table across all folds
coef_df = pd.concat(coef_stats, ignore_index=True)
coef_df.to_parquet(COEF_STATS_PATH, index=False)
print(f"done -> {OUTPUT_PATH}")
print(f"coef stats: {coef_df.shape[0]:,} rows across {coef_df['model_year'].nunique()} "
      f"models -> {COEF_STATS_PATH}")


In [ ]:
preds = pd.read_parquet(OUTPUT_PATH)
print(preds.shape, preds.columns.tolist())
preds.head()

In [ ]:
# ---- preview the stored coefficient statistics -----------------------------
cstats = pd.read_parquet(COEF_STATS_PATH)
print(cstats.shape, cstats.columns.tolist())
# most recent model, strongest features by |t|
_last = cstats["model_year"].max()
(cstats[cstats["model_year"] == _last]
        .reindex(cstats[cstats["model_year"] == _last]["t_stat"].abs().sort_values(ascending=False).index)
        .head(12).round(4))
